# **5G Deployment Example with OCUDU ZMQ (1)**

In this example, we will deploy a 5G network with:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OCUDU gNB with ZeroMQ
- SRSUE with ZeroMQ as simulated UE

Reference: https://docs.ocudu.org/tutorials/srsue/#zeromq-based-setup

First, we install the required dependencies:

## Install Dependencies:

Install the `ifran` library:

In [ ]:
! pip install ifran

Install the Ansible dependencies:

In [ ]:
! ansible-galaxy install -r ../../ansible/requirements.yaml

Install the SLICES CLI used for the BI to create VMs:

In [ ]:
! pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
! slices auth login

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

In [1]:
from ifran import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  sim_ru_gnb:
    txGainDB: 0
    rxGainDB: 0

  sim_ru_ue:
    txGainDB: 50
    rxGainDB: 40

ran:
  components:
    gnb:
      type: gnb
      config:
        ru: sim_ru_gnb
        radioConfiguration:
          nrBand: 3
          downlinkArfcn: 368500
          channelBandwidthMhz: 10
          subcarrierSpacingKhz: 15
          ssbPeriodicityMs: 10
          prachConfigurationIndex: 1

ues:
  sim_ue:
    config:
      nrBands: [3]
      ru: sim_ru_ue

slices:
  internet:
    sst: 1
    sd: 0xFFFFFF # explicitly set sd to avoid core regection
"""
)

In [2]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    project: post5g-beta
    experiment: ifran
    flavor: xlarge # ocudu zmq needs more than 5 CPU cores (see: https://github.com/srsran/srsRAN_Project/issues/1472)

ran:
  implementation: ocudu
  components:
    gnb:
      type: gnb
      advanced_config:
        inactivity_timer: 7200
        srate: 11.52
        base_srate: "11.52e6"
        pdcch:
          common:
            ss0_index: 0
            coreset0_index: 6
          dedicated:
            ss2_type: common
            dci_format_0_1_and_1_1: false
        pdsch:
          mcs_table: qam64
        pusch:
          mcs_table: qam64

ues:
  sim_ue:
    type: containerized
    implementation: srsue
    advanced_config:
      srate: "11.52e6"
"""
)

## Provisioning:

In [3]:
from ifran import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [4]:
await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select projec

## Kubernetes Deployment:

In [5]:
from ifran import K8SManagement

k8s = K8SManagement(prov)

In [6]:
await k8s.deploy()


PLAY [node4] *******************************************************************

PLAY [node3] *******************************************************************

PLAY [node1] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node4'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node3'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node1'] ****************

PLAY [node2] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node2'] ****************
changed: [node2]

TASK [k8s/prepare : Install base packages required for Kubernetes on 'node2'] ***
changed: [node3]

TASK [k8s/prepare : Install base packages required for Kubernetes on 'node3'] ***
changed: [node1]

TASK [k8s/prepare : Install base packages required for Kubernetes on 'node1'] ***
changed: [node4]

TASK [k8s/prepare : Instal

## 5G Core Deployment:

In [7]:
from ifran import CoreManagement

core = CoreManagement(k8s)

In [8]:
await core.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-ausf' chart] *****

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-udm' chart] ******

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nrf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

PLAY [node1] 

## RAN Deployment:

In [9]:
from ifran import RANManagement

ran = RANManagement(core)

In [10]:
await ran.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'ocudu-gnb' chart] ********
changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'ocudu-gnb' if provided on localhost] ***
skipping: [node1]

TASK [helm/prepare : Compress built-in Helm chart directory for 'ocudu-gnb' on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Extract Helm chart archive for 'ocudu-gnb' from localhost] ***
changed: [node1]

TASK [helm/prepare : Remove temporary Helm chart archive for 'ocudu-gnb' from localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'ocudu-gnb'] ***
changed: [node1]

TASK [helm/prepare : Render custom templated Helm files for 'ocudu-gnb' if provided] ***
skipping: [node1]

TASK [helm/prepare : Render built-in templated Helm files for 'ocudu-gnb'] *****
changed: [node1] => (item=values.yaml)
changed: 

Get OCUDU ZMQ gNB logs:

In [11]:
print(await ran.get_logs(components=["gnb"], quiet=True))


--== OCUDU gNB (commit 050a2bb) ==--

2026-11-27T20:04:05.007097 [GNB     ] [I] Built in Release mode using commit 050a2bb on branch HEAD
2026-11-27T20:04:05.009956 [CONFIG  ] [D] Input configuration (all values): 
gnb_id: 1
gnb_id_bit_length: 32
ran_node_name: ocucp01
metrics:
  enable_log: false
  enable_json: false
  enable_verbose: false
  layers:
    enable_app_usage: false
    enable_executor: false
    enable_ngap: false
    enable_pdcp: false
    enable_rrc: false
    enable_e1ap: false
    enable_nrup_cu: false
    skip_cu_up_executor: true
    enable_sched: true
    enable_rlc: false
    enable_mac: false
    enable_du_low: false
    enable_ru: false
  periodicity:
    app_usage_report_period: 1000
    executors_report_period: 1000
    cu_cp_report_period: 1000
    cu_up_report_period: 1000
    du_report_period: 1000
  autostart_stdout_metrics: true
buffer_pool:
  nof_segments: 1048576
  segment_size: 2048
log:
  filename: stdout
  lib_level: warning
  e2ap_level: info
  con

## UE Deployement:

In [12]:
from ifran import UEManagement

ue = UEManagement(ran)

In [13]:
await ue.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'srsue-sim-ue' chart] *****
changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'srsue-sim-ue' if provided on localhost] ***
skipping: [node1]

TASK [helm/prepare : Compress built-in Helm chart directory for 'srsue-sim-ue' on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Extract Helm chart archive for 'srsue-sim-ue' from localhost] ***
changed: [node1]

TASK [helm/prepare : Remove temporary Helm chart archive for 'srsue-sim-ue' from localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'srsue-sim-ue'] ***
changed: [node1]

TASK [helm/prepare : Render custom templated Helm files for 'srsue-sim-ue' if provided] ***
skipping: [node1]

TASK [helm/prepare : Render built-in templated Helm files for 'srsue-sim-ue'] ***
changed: [node1] => (item=val

Get SRSUE ZMQ UE logs:

In [14]:
print(await ue.get_logs(components=["sim_ue"], quiet=True))

2026-11-27T20:05:14.199528 [PHY_LIB] [I] [    0]  PDCCH DMRS Rx: Measure L=2; ncce=0; RSRP=-inf dBfs; EPRE=-inf dBfs; Corr=0.000
2026-11-27T20:05:14.199529 [PHY_LIB] [I] [    0]  Discarded PDCCH candidate L=2;ncce=0; Invalid measurement;
2026-11-27T20:05:14.199530 [PHY_LIB] [I] [    0]  PDCCH DMRS Rx: Measure L=2; ncce=0; RSRP=-inf dBfs; EPRE=-inf dBfs; Corr=0.000
2026-11-27T20:05:14.199531 [PHY_LIB] [I] [    0]  Discarded PDCCH candidate L=2;ncce=0; Invalid measurement;
2026-11-27T20:05:14.199532 [PHY_LIB] [I] [    0]  PDCCH DMRS Rx: Measure L=2; ncce=4; RSRP=-inf dBfs; EPRE=-inf dBfs; Corr=0.000
2026-11-27T20:05:14.199532 [PHY_LIB] [I] [    0]  Discarded PDCCH candidate L=2;ncce=4; Invalid measurement;
2026-11-27T20:05:14.199533 [PHY_LIB] [I] [    0]  PDCCH DMRS Rx: Measure L=2; ncce=4; RSRP=-inf dBfs; EPRE=-inf dBfs; Corr=0.000
2026-11-27T20:05:14.199533 [PHY_LIB] [I] [    0]  Discarded PDCCH candidate L=2;ncce=4; Invalid measurement;
2026-11-27T20:05:14.199535 [PHY_LIB] [I] [    0]

Get gNB logs after UE was connected:

In [15]:
print(await ran.get_logs(components=["gnb"], quiet=True))


--== OCUDU gNB (commit 050a2bb) ==--

2026-11-27T20:04:05.007097 [GNB     ] [I] Built in Release mode using commit 050a2bb on branch HEAD
2026-11-27T20:04:05.009956 [CONFIG  ] [D] Input configuration (all values): 
gnb_id: 1
gnb_id_bit_length: 32
ran_node_name: ocucp01
metrics:
  enable_log: false
  enable_json: false
  enable_verbose: false
  layers:
    enable_app_usage: false
    enable_executor: false
    enable_ngap: false
    enable_pdcp: false
    enable_rrc: false
    enable_e1ap: false
    enable_nrup_cu: false
    skip_cu_up_executor: true
    enable_sched: true
    enable_rlc: false
    enable_mac: false
    enable_du_low: false
    enable_ru: false
  periodicity:
    app_usage_report_period: 1000
    executors_report_period: 1000
    cu_cp_report_period: 1000
    cu_up_report_period: 1000
    du_report_period: 1000
  autostart_stdout_metrics: true
buffer_pool:
  nof_segments: 1048576
  segment_size: 2048
log:
  filename: stdout
  lib_level: warning
  e2ap_level: info
  con

Veify that the UE got a PDU session:

In [16]:
print(await ue.run_command(command=["ip", "addr"], quiet=True))

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0@if11: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9166 qdisc noqueue state UP group default qlen 1000
    link/ether 7e:25:91:2d:e9:4c brd ff:ff:ff:ff:ff:ff link-netnsid 0
    inet 10.244.223.54/32 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::7c25:91ff:fe2d:e94c/64 scope link 
       valid_lft forever preferred_lft forever
3: tun_srsue: <POINTOPOINT,MULTICAST,NOARP,UP,LOWER_UP> mtu 1500 qdisc fq_codel state UNKNOWN group default qlen 500
    link/none 
    inet 11.1.1.2/24 scope global tun_srsue
       valid_lft forever preferred_lft forever



#### Test connectivity:

In [17]:
print(await ue.run_command(command=["apt", "update"], quiet=True))

Get:1 http://archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2277 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble/universe amd64 Packages [19.3 MB]
Get:7 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1563 kB]
Get:8 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1514 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [55.6 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble/multiverse amd64 Packages [331 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble/main amd64 Packages [1808 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble/restricted amd64 Packages [117 kB]
Get:13 http://archive.ubuntu.com/ubuntu nob

In [18]:
print(await ue.run_command(command=["apt", "install", "-y", "iputils-ping"], quiet=True))

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  iputils-ping
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 44.6 kB of archives.
After this operation, 124 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 iputils-ping amd64 3:20240117-1ubuntu0.1 [44.6 kB]
Fetched 44.6 kB in 30s (1478 B/s)
Selecting previously unselected package iputils-ping.
(Reading database ... 35148 files and directories currently installed.)
Preparing to unpack .../iputils-ping_3%3a20240117-1ubuntu0.1_amd64.deb ...
Unpacking iputils-ping (3:20240117-1ubuntu0.1) ...
Setting up iputils-ping (3:20240117-1ubuntu0.1) ...



In [19]:
print(await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "tun_srsue"], quiet=True))

PING 11.1.1.1 (11.1.1.1) from 11.1.1.2 tun_srsue: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=31.8 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=58.6 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=35.8 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=28.7 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=20.4 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=35.9 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5007ms
rtt min/avg/max/mdev = 20.377/35.204/58.575/11.691 ms



In [20]:
print((await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "tun_srsue"], quiet=True)))

PING 8.8.8.8 (8.8.8.8) from 11.1.1.2 tun_srsue: 56(84) bytes of data.
64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=40.0 ms
64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=33.1 ms
64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=22.6 ms
64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=37.4 ms
64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=32.1 ms
64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=25.9 ms

--- 8.8.8.8 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5005ms
rtt min/avg/max/mdev = 22.580/31.853/40.023/6.071 ms



In [21]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

(MetaModel(core=Core(components={'amf': AMFComponent(category=<Category5G.CORE: 'core'>, type=<CoreComponentType.AMF: 'amf'>, config=AMFConfig(sbi=SBI(interfaceName='eth0', ipv4EndPointAddress=IPv4Address('0.0.0.0'), ipv6EndPointAddress=None, port=80, protocol='TCP', scheme='http', apiVersion='v1', tls=TLS(enabled=False, certificate=None, privateKey=None)), nrfUri=None, n2=NGAP(interfaceName='eth0', ipv4EndPointAddress=IPv4Address('0.0.0.0'), ipv6EndPointAddress=None, port=38412, protocol='SCTP'), guamiList=[GUAMI(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), amfRegionId=2, amfSetId=1, amfPointer=0, amfId=131136)], taiList=[TAI(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), tac=1)], plmnList=[PLMNSupportItem(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), snssaiList=['internet'])], securityAlgorithmConfiguration=SecurityAlgorithmConfiguration(integrityOrder=[<IntegrityAlgorithm.NIA2: 'nia2'>, <IntegrityAlgorithm.NIA1: 'nia1'>, <IntegrityAlgorithm.NIA0: 'nia0'

## Test OCUDU ZMQ gNB with OAI NR UE:

Reference: https://docs.ocudu.org/tutorials/oaiue/#zeromq-based-setup

#### Add a second OCUDU gNB:

In [22]:
metamodel, deployment_model = ran.output()

In [23]:
from ifran import RU

metamodel.rus["sim_ru_gnb2"] = RU(txGainDB=-12, rxGainDB=0)

In [24]:
ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

In [25]:
from metamodel_3gpp.models.ran import GNBComponent
from ifran.models.ran import DeployedRANComponent

ran.add_component(
    name="gnb2",
    component_meta=GNBComponent.from_yaml(
"""
type: gnb
config:
  nrCellId: 2
  physicalCellId: 1
  ru: sim_ru_gnb2
  radioConfiguration:
    nrBand: 78
    downlinkArfcn: 632628
    ssbArfcn: 632256
    pointAFrequencyArfcn: 632016
    channelBandwidthMhz: 20
    prachConfigurationIndex: 159
    subcarrierSpacingKhz: 30
    ssbPeriodicityMs: 10
    tddConfiguration:
      periodicityMilliseconds: 2.5
      nrofDownlinkSlots: 3
      nrofDownlinkSymbols: 10
      nrofUplinkSlots: 1
      nrofUplinkSymbols: 2
"""
    ),
    component_deployment=DeployedRANComponent.from_yaml(
"""
type: gnb
implementation: ocudu
advanced_config:
  srate: 23.04
  csi:
    csi_rs_enabled: true
  pucch:
    nof_cell_csi_res: 1
"""
    ),
)

Deploy `gnb2`:

In [26]:
await ran.deploy(components=["gnb2"])


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'ocudu-gnb2' chart] *******
changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'ocudu-gnb2' if provided on localhost] ***
skipping: [node1]

TASK [helm/prepare : Compress built-in Helm chart directory for 'ocudu-gnb2' on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Extract Helm chart archive for 'ocudu-gnb2' from localhost] ***
changed: [node1]

TASK [helm/prepare : Remove temporary Helm chart archive for 'ocudu-gnb2' from localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'ocudu-gnb2'] ***
changed: [node1]

TASK [helm/prepare : Render custom templated Helm files for 'ocudu-gnb2' if provided] ***
skipping: [node1]

TASK [helm/prepare : Render built-in templated Helm files for 'ocudu-gnb2'] ****
changed: [node1] => (item=values.yaml)
cha

Check its logs:

In [27]:
print(await ran.get_logs(components=["gnb2"], quiet=True))


--== OCUDU gNB (commit 050a2bb) ==--

2026-11-27T20:07:25.246102 [GNB     ] [I] Built in Release mode using commit 050a2bb on branch HEAD
2026-11-27T20:07:25.249408 [CONFIG  ] [D] Input configuration (all values): 
gnb_id: 1
gnb_id_bit_length: 32
ran_node_name: ocucp01
metrics:
  enable_log: false
  enable_json: false
  enable_verbose: false
  layers:
    enable_app_usage: false
    enable_executor: false
    enable_ngap: false
    enable_pdcp: false
    enable_rrc: false
    enable_e1ap: false
    enable_nrup_cu: false
    skip_cu_up_executor: true
    enable_sched: true
    enable_rlc: false
    enable_mac: false
    enable_du_low: false
    enable_ru: false
  periodicity:
    app_usage_report_period: 1000
    executors_report_period: 1000
    cu_cp_report_period: 1000
    cu_up_report_period: 1000
    du_report_period: 1000
  autostart_stdout_metrics: true
buffer_pool:
  nof_segments: 1048576
  segment_size: 2048
log:
  filename: stdout
  lib_level: warning
  e2ap_level: info
  con

Add the OAI NR UE to be connected to OCUDU `gnb2` over ZeroMQ:

In [28]:
ue = UEManagement(ran)

In [29]:
from metamodel_3gpp.models.ue import UE
from ifran.models.ue import DeployedUE

ue.add_component(
    name="sim_ue2",
    component_meta=UE.from_yaml(
"""
config:
  supi: "001010000000101"
  ru: sim_ru_gnb
  gnbs:
    - gnb2
"""
),
    component_deployment=DeployedUE.from_yaml(
"""
type: containerized
implementation: oai
"""
),
)

Add it to the 5G core database:

In [30]:
core = CoreManagement(ue)
await core.register_ues(ues=["sim_ue2"])


PLAY [node1] *******************************************************************

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue2' to Open5GS database] ***
[WARNING]: Deprecation warnings can be disabled by setting `deprecation_warnings=False` in ansible.cfg.
[DEPRECATION WARNING]: Importing 'to_native' from 'ansible.module_utils._text' is deprecated. This feature will be removed from ansible-core version 2.24. Use ansible.module_utils.common.text.converters instead.
changed: [node1]

PLAY RECAP *********************************************************************
node1                      : ok=1    changed=1    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   


Deploy it:

In [31]:
await ue.deploy(components=["sim_ue2"])


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue2' chart] ******
changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'oai-sim-ue2' if provided on localhost] ***
skipping: [node1]

TASK [helm/prepare : Compress built-in Helm chart directory for 'oai-sim-ue2' on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Extract Helm chart archive for 'oai-sim-ue2' from localhost] ***
changed: [node1]

TASK [helm/prepare : Remove temporary Helm chart archive for 'oai-sim-ue2' from localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'oai-sim-ue2'] ***
changed: [node1]

TASK [helm/prepare : Render custom templated Helm files for 'oai-sim-ue2' if provided] ***
skipping: [node1]

TASK [helm/prepare : Render built-in templated Helm files for 'oai-sim-ue2'] ***
changed: [node1] => (item=values.yam

Check it's logs:

In [32]:
print(await ue.get_logs(components=["sim_ue2"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--ue-scan-carrier" "--thread-pool" "-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1" "--uecap_file" "/opt/oai-nr-ue/etc/uecap_ports1.xml" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffff

Check if it got a PDU session:

In [37]:
print(await ue.run_command(command=["ip", "addr"], components=["sim_ue2"], quiet=True))

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0@if11: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9166 qdisc noqueue state UP group default qlen 1000
    link/ether a2:22:09:bd:61:a4 brd ff:ff:ff:ff:ff:ff link-netnsid 0
    inet 10.244.80.136/32 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::a022:9ff:febd:61a4/64 scope link 
       valid_lft forever preferred_lft forever
3: oaitun_ue1: <POINTOPOINT,NOARP,UP,LOWER_UP> mtu 1500 qdisc fq_codel state UNKNOWN group default qlen 500
    link/none 
    inet 11.1.1.3/24 scope global oaitun_ue1
       valid_lft forever preferred_lft forever
    inet6 fe80::718e:60f8:60d7:428d/64 scope link stable-privacy 
       valid_lft forever preferred_lft forever



Check connectivity:

In [34]:
print(await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "oaitun_ue1"], components=["sim_ue2"], quiet=True))

PING 11.1.1.1 (11.1.1.1) from 11.1.1.3 oaitun_ue1: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=43.0 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=72.0 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=41.7 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=66.6 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=76.0 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=49.4 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5005ms
rtt min/avg/max/mdev = 41.729/58.117/76.004/13.894 ms



In [35]:
print(await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "oaitun_ue1"], components=["sim_ue2"], quiet=True))

PING 8.8.8.8 (8.8.8.8) from 11.1.1.3 oaitun_ue1: 56(84) bytes of data.
64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=44.2 ms
64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=36.5 ms
64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=62.7 ms
64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=56.2 ms
64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=69.9 ms
64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=38.8 ms

--- 8.8.8.8 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5005ms
rtt min/avg/max/mdev = 36.530/51.382/69.882/12.411 ms



In [36]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

(MetaModel(core=Core(components={'amf': AMFComponent(category=<Category5G.CORE: 'core'>, type=<CoreComponentType.AMF: 'amf'>, config=AMFConfig(sbi=SBI(interfaceName='eth0', ipv4EndPointAddress=IPv4Address('0.0.0.0'), ipv6EndPointAddress=None, port=80, protocol='TCP', scheme='http', apiVersion='v1', tls=TLS(enabled=False, certificate=None, privateKey=None)), nrfUri=None, n2=NGAP(interfaceName='eth0', ipv4EndPointAddress=IPv4Address('0.0.0.0'), ipv6EndPointAddress=None, port=38412, protocol='SCTP'), guamiList=[GUAMI(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), amfRegionId=2, amfSetId=1, amfPointer=0, amfId=131136)], taiList=[TAI(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), tac=1)], plmnList=[PLMNSupportItem(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), snssaiList=['internet'])], securityAlgorithmConfiguration=SecurityAlgorithmConfiguration(integrityOrder=[<IntegrityAlgorithm.NIA2: 'nia2'>, <IntegrityAlgorithm.NIA1: 'nia1'>, <IntegrityAlgorithm.NIA0: 'nia0'